In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import logging
from lattice import Lattice, AsyncLLM, SyncLLM
from typing import Dict, List
from datetime import datetime
from dotenv import load_dotenv
import json
from pathlib import Path


logging.basicConfig(level=logging.INFO)
load_dotenv()

True

In [6]:
def read_json_l(file_path: Path) -> List[Dict]:
    with open(file_path, "r") as file:
        return [json.loads(line) for line in file]
        
def process_claude_code_data(name: str, file_dir: str | None = None, min_length: int = 30):
    name = "Dora"
    if file_dir is None:
        claude_code_dir = Path.home() / ".claude" / "projects"
    else:
        claude_code_dir = Path(file_dir)
    interaction_traces = []
    for directory in claude_code_dir.iterdir():
        for file in directory.iterdir():
            if file.suffix == ".jsonl":
                data = read_json_l(file)
                session_trace = []
                for message in data:
                    if 'message' in message:
                        msg = message['message']
                        if msg['role'] == 'user':
                            session_trace.append({
                                "interaction": f"{name}: {msg['content']}",
                                "metadata": {}
                            })
                        else:
                            session_trace.append({
                                "interaction": f"Claude Code: {msg['content']}",
                                "metadata": {}
                            })
                if len(session_trace) > min_length:
                    m_time = os.path.getmtime(file)
                    interaction_traces.append({
                        "interactions": session_trace,
                        "time": datetime.fromtimestamp(m_time).strftime('%Y-%m-%d %H:%M:%S')
                    })
    return interaction_traces


In [9]:
config = {
    0: {
        "type": "session",
        "value": "5"
    },
    1: {
        "type": "session",
        "value": "10"
    },
}

name = "Dora"
interaction_traces = process_claude_code_data(name)

In [ ]:
l = Lattice(
    name="Dora",
    interactions=interaction_traces,
    insight_model=AsyncLLM(name="claude-opus-4-6", api_key=os.getenv("ANTHROPIC_API_KEY")),
    observer_model=AsyncLLM(name="claude-sonnet-4-6", api_key=os.getenv("ANTHROPIC_API_KEY")),
    evidence_model=AsyncLLM(name="claude-sonnet-4-6", api_key=os.getenv("ANTHROPIC_API_KEY")),
    format_model=SyncLLM(name="claude-sonnet-4-6", api_key=os.getenv("ANTHROPIC_API_KEY")),
    params={"max_concurrent": 100, "min_insights": 3, "window_size": 100},
    description="the user's conversation with Claude Code, an AI-based coding agent"
)

In [11]:
await l.build(config)

INFO:Lattice:Building lattice with config: {0: {'type': 'session', 'value': '5'}, 1: {'type': 'session', 'value': '10'}}
INFO:Lattice:Making observations
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTT

In [13]:
l.print_layer()

[0] Vision-Driven Planner, Hands-Off Executor
     Dora arrives at sessions with obsessive preparation — precise hex codes, TypeScript interfaces, multi-thousand-word prompts, and a clear mental model — yet the moment execution begins, she hands off entirely. She does not attempt to diagnose errors independently, instead relaying raw outputs and deferring to others for analysis. This reveals a sharp cognitive boundary: someone who pre-specifies exact API endpoints cannot be bothered to independently interpret a basic environment error. Her energy is concentrated upstream, in definition and design, not downstream in debugging or iteration.
     Context: Most relevant when Dora is setting up a new project, briefing collaborators, or scoping work — she will excel at requirements definition. In debugging, troubleshooting, or iterative execution contexts, she will rely heavily on others to drive resolution.
     Metadata: {'input_session': 0, 'time': '2026-03-31 23:51:40'}

[1] Architectura

In [14]:
l.visualize()

In [15]:
l.save("claude_code_lattice.json")